# BeyondSmile: A Challenge on Detecting Depression through Facial Behavior and Head Gestures

In [24]:
import pandas as pd
import numpy as np
import tsfel
import neurokit2 as nk
import matplotlib.pyplot as plt
import datetime
import json
import pycatch22


# Read the colums for the data

In [25]:
columns_tself = pd.read_csv('./columns_mid_tsfel_prob_eye.csv')

In [26]:
del columns_tself['Unnamed: 0']

In [27]:
tself_columns_mid = list(columns_tself.columns)


In [28]:
columns_pycatch = pd.read_csv('./columns_mid_pycatch_prob_eye.csv')

In [29]:
del columns_pycatch['Unnamed: 0']

In [30]:
pycatch_columns_mid = list(columns_pycatch.columns)


In [31]:
tself_columns_mor = [name.replace('mid', 'mor') for name in tself_columns_mid]
tself_columns_aft = [name.replace('mid', 'aft') for name in tself_columns_mid]
tself_columns_eve = [name.replace('mid', 'eve') for name in tself_columns_mid]

In [32]:
pycatch_columns_mor = [name.replace('mid', 'mor') for name in pycatch_columns_mid]
pycatch_columns_aft = [name.replace('mid', 'aft') for name in pycatch_columns_mid]
pycatch_columns_eve = [name.replace('mid', 'eve') for name in pycatch_columns_mid]

# Read the labels


In [103]:
record = 27
data_phq = pd.read_csv('./dataset/groundtruth/phq9 _date.csv')
data_patient_depression = data_phq.loc[record]
patient = data_phq.loc[record]['pid']
diagnosis = data_phq.loc[record]['depression_episode']
start_monitoring = data_patient_depression.start_ts
end_monitoring = data_patient_depression.end_ts
#Change the dates into the timestamp
element_start = datetime.datetime(2022, int(start_monitoring.split('/')[0]), int(start_monitoring.split('/')[1]))
timestamp_start = datetime.datetime.timestamp(element_start)
element_end = datetime.datetime(2022, int(end_monitoring.split('/')[0]), int(end_monitoring.split('/')[1]))
timestamp_end = datetime.datetime.timestamp(element_end)
#Reorder the data
with open('./dataset/data/'+patient+ '.json', 'r') as f:
        data = json.load(f)

In [104]:
data_phq

,pid,start_ts,end_ts,start_phq9,end_phq9,depression_episode
0,P08,7/21/22,08/09/2022,6,1.0,0
1,P08,08/09/2022,8/23/22,1,9.0,0
2,P10,7/21/22,08/09/2022,8,7.0,1
3,P10,08/09/2022,09/02/2022,7,2.0,0
4,P12,7/22/22,08/09/2022,10,12.0,1
5,P12,08/09/2022,8/23/22,12,9.0,1
6,P13,7/25/22,08/09/2022,1,3.0,0
7,P13,08/09/2022,8/23/22,3,2.0,0
8,P14,7/25/22,08/08/2022,11,NaN,0
9,P15,7/26/22,08/10/2022,4,9.0,0


In [105]:
times = []
numbers = []
for i in range(0, len(data)):
    times.append(int(data[i]['timestamp'])/1000)
    numbers.append(i)
min_value = datetime.datetime.fromtimestamp(min(times)).isoformat()
max_value = datetime.datetime.fromtimestamp(max(times)).isoformat()

In [106]:
b = enumerate(times)
c = sorted(b, key = lambda i:i[1])
times_index_primary = []

for e in c:
    times_index_primary.append(e[0])

sorted_times = sorted(times)


# Reorder data json

In [107]:
data2 = []
for i in range(0, len(times_index_primary)):
    data2.append(data[times_index_primary[i]])

In [108]:
times = []
numbers = []
for i in range(0, len(data2)):
    times.append(int(data2[i]['timestamp'])/1000)
    numbers.append(i)

# Find the beginning of the record and end of the record

In [109]:
counter_start = 0
while(sorted_times[counter_start]<timestamp_start):
    counter_start +=1

In [110]:
counter_end = counter_start
for i in range(counter_start, len(sorted_times)):
    if sorted_times[counter_end]<=timestamp_end:
        counter_end +=1
    else:
        break
counter_end = counter_end - 1

In [111]:
#Select subdataset
data3 = data2[counter_start:counter_end+1]

# Select the day for 4 periods: midnight (12am-6am), morning (6am-12pm), afternoon (12pm-6pm), and evening (6pm12am) (to daytime!)

In [112]:
midnight_time = []
morning_time = []
afternoon_time = []
evening_time = []
for i in range(0, len(data3)):
    hour_sample = datetime.datetime.fromtimestamp(float(data3[i]['timestamp'])/1000).hour
    if hour_sample>=0 and hour_sample<6:
        midnight_time.append(i)
    if hour_sample>=6 and hour_sample<12:
        morning_time.append(i)
    if hour_sample>=12 and hour_sample<18:
        afternoon_time.append(i)
    if hour_sample>=18 and hour_sample<=23:
        evening_time.append(i)
        

In [113]:
data_midnight = []
for i in range(0, len(midnight_time)):
    data_midnight.append(data3[midnight_time[i]])

In [114]:
data_morning = []
for i in range(0, len(morning_time)):
    data_morning.append(data3[morning_time[i]])

In [115]:
data_afternoon = []
for i in range(0, len(afternoon_time)):
    data_afternoon.append(data3[afternoon_time[i]])

In [116]:
data_evening = []
for i in range(0, len(evening_time)):
    data_evening.append(data3[evening_time[i]])

# Define separete subdata for the midningt, morning, afternoon and evening 

# Extract midnight featues for smiling and open eyes probabilities

In [117]:
COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
COLUMN_NAMES_mid = []
for i in range(0, len(COLUMN_NAMES)):
    COLUMN_NAMES_mid.append(COLUMN_NAMES[i] +'_mid')
records_prob_mid = pd.DataFrame(columns= COLUMN_NAMES_mid)

for i in range(0, len(data_midnight)):
    prob_data = data_midnight[i]['classification']
    prob_values = list(prob_data.values())

    if len(prob_data)!=0:
        records_prob_mid.loc[i] = prob_values
    else: 
        records_prob_mid.loc[i] = [np.nan]*3

In [118]:
records_prob_mid

,lefteye_mid,righteye_mid,smiling_mid
0,0.990354,0.997131,0.137512


In [119]:
records_prob_mid_cleaned = records_prob_mid.copy()
records_prob_mid_cleaned = records_prob_mid_cleaned.dropna()

In [120]:
records_prob_mid_cleaned

,lefteye_mid,righteye_mid,smiling_mid
0,0.990354,0.997131,0.137512


In [121]:
if len(records_prob_mid_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
    cfg = tsfel.get_features_by_domain()

    # Extract features
    X = tsfel.time_series_features_extractor(cfg, records_prob_mid_cleaned)
    X_mid_to_delete = []
    for name in X.columns:
        if 'Spectrogram mean coefficient_' in name:
            X_mid_to_delete.append(name)
    X = X.drop(X_mid_to_delete, axis=1)
            
else:
    X = pd.DataFrame(columns=tself_columns_mid)
    X.loc[0] = [np.nan]*372

In [122]:
data_prob_mid = X.copy()

In [123]:
if len(records_prob_mid_cleaned)>3:     
    for j in range(0, len(COLUMN_NAMES_mid)):
        name_prob = COLUMN_NAMES_mid[j]
        features_pycatch = pycatch22.catch22_all(records_prob_mid_cleaned[name_prob])
        COLUMN = []
        for i in range(0, len(features_pycatch['names'])):
            COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
        features_prob_sub_mid = pd.DataFrame(columns=COLUMN)
        features_prob_sub_mid.loc[0] = features_pycatch['values']
        if j == 0:
            features_prob_mid = features_prob_sub_mid.copy()
        else:
            features_prob_mid = pd.concat([features_prob_mid, features_prob_sub_mid], axis=1)
else:
    features_prob_mid = pd.DataFrame(columns=pycatch_columns_mid)
    features_prob_mid.loc[0] = [np.nan]*66 



In [124]:
data_prob_mid = pd.concat([data_prob_mid, features_prob_mid], axis=1)

In [125]:
data_prob_mid

,lefteye_mid_Absolute energy,lefteye_mid_Area under the curve,lefteye_mid_Autocorrelation,lefteye_mid_Average power,lefteye_mid_Centroid,lefteye_mid_ECDF Percentile Count_0,lefteye_mid_ECDF Percentile Count_1,lefteye_mid_ECDF Percentile_0,lefteye_mid_ECDF Percentile_1,lefteye_mid_ECDF_0,...,smiling_mid_FC_LocalSimple_mean1_tauresrat,smiling_mid_DN_OutlierInclude_p_001_mdrmd,smiling_mid_DN_OutlierInclude_n_001_mdrmd,smiling_mid_SP_Summaries_welch_rect_area_5_1,smiling_mid_SB_BinaryStats_diff_longstretch0,smiling_mid_SB_MotifThree_quantile_hh,smiling_mid_SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1,smiling_mid_SC_FluctAnal_2_dfa_50_1_2_logi_prop_r1,smiling_mid_SP_Summaries_welch_rect_centroid,smiling_mid_FC_LocalSimple_mean3_stderr
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [126]:
approx_entropy_columns = [name + '_app_ent' for name in records_prob_mid_cleaned.columns]
data_approx_entropy_mid = pd.DataFrame(columns=approx_entropy_columns)
app_ent_prob_mid= []



In [127]:
for i in range(0, len(approx_entropy_columns)): 
    try:
        approximate_entropy, parameters = nk.entropy_approximate(records_prob_mid_cleaned[records_prob_mid_cleaned.columns[i]])
         # Approximate entropy
    except:
        approximate_entropy = 0
    app_ent_prob_mid.append(approximate_entropy)
data_approx_entropy_mid.loc[0] = app_ent_prob_mid

In [128]:
data_prob_mid = pd.concat([data_prob_mid, data_approx_entropy_mid], axis=1)

In [129]:
rsd_columns_mid = [name + '_rsd' for name in records_prob_mid_cleaned.columns]
data_rsd_mid = pd.DataFrame(columns=rsd_columns_mid)
rsd_prob_mid = []
for i in range(0, len(rsd_columns_mid)): 
    rsd = 100*np.std(records_prob_mid_cleaned[records_prob_mid_cleaned.columns[i]])/(np.mean(records_prob_mid_cleaned[records_prob_mid_cleaned.columns[i]])+0.00000000000000000000001)
    rsd_prob_mid.append(rsd)
data_rsd_mid.loc[0] = rsd_prob_mid


In [130]:
data_prob_mid = pd.concat([data_prob_mid, data_rsd_mid], axis=1)

# For morning

In [131]:
COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
COLUMN_NAMES_mor = []
for i in range(0, len(COLUMN_NAMES)):
    COLUMN_NAMES_mor.append(COLUMN_NAMES[i] +'_mor')
records_prob_mor = pd.DataFrame(columns= COLUMN_NAMES_mor)

for i in range(0, len(data_morning)):
    prob_data = data_morning[i]['classification']
    prob_values = list(prob_data.values())

    if len(prob_data)!=0:
        records_prob_mor.loc[i] = prob_values
    else: 
        records_prob_mor.loc[i] = [np.nan]*3

In [132]:
records_prob_mor_cleaned = records_prob_mor.copy()
records_prob_mor_cleaned = records_prob_mor_cleaned.dropna()

In [133]:
if len(records_prob_mor_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
    cfg = tsfel.get_features_by_domain()

    # Extract features
    X = tsfel.time_series_features_extractor(cfg, records_prob_mor_cleaned)
    X_mor_to_delete = []
    for name in X.columns:
        if 'Spectrogram mean coefficient_' in name:
            X_mor_to_delete.append(name)
    X = X.drop(X_mor_to_delete, axis=1)
            
else:
    X = pd.DataFrame(columns=tself_columns_mor)
    X.loc[0] = [np.nan]*372

In [134]:
data_prob_mor = X.copy()

In [135]:
if len(records_prob_mor_cleaned)>3:    
    for j in range(0, len(COLUMN_NAMES_mor)):
        name_prob = COLUMN_NAMES_mor[j]
        features_pycatch = pycatch22.catch22_all(records_prob_mor_cleaned[name_prob])
        COLUMN = []
        for i in range(0, len(features_pycatch['names'])):
            COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
        features_prob_sub_mor = pd.DataFrame(columns=COLUMN)
        features_prob_sub_mor.loc[0] = features_pycatch['values']
        if j == 0:
            features_prob_mor = features_prob_sub_mor.copy()
        else:
            features_prob_mor = pd.concat([features_prob_mor, features_prob_sub_mor], axis=1)
else:
    features_prob_mor = pd.DataFrame(columns=pycatch_columns_mor)
    features_prob_mor.loc[0] = [np.nan]*66 



In [136]:
data_prob_mor = pd.concat([data_prob_mor, features_prob_mor], axis=1)

In [137]:
approx_entropy_columns = [name + '_app_ent' for name in records_prob_mor_cleaned.columns]
data_approx_entropy_mor = pd.DataFrame(columns=approx_entropy_columns)
app_ent_prob_mor= []



In [138]:
for i in range(0, len(approx_entropy_columns)): 
    try:
        approximate_entropy, parameters = nk.entropy_approximate(records_prob_mor_cleaned[records_prob_mor_cleaned.columns[i]])
         # Approximate entropy
    except:
        approximate_entropy = 0
    app_ent_prob_mor.append(approximate_entropy)
data_approx_entropy_mor.loc[0] = app_ent_prob_mor

In [139]:
data_prob_mor = pd.concat([data_prob_mor, data_approx_entropy_mor], axis=1)

In [140]:
rsd_columns_mor= [name + '_rsd' for name in records_prob_mor_cleaned.columns]
data_rsd_mor = pd.DataFrame(columns=rsd_columns_mor)
rsd_prob_mor = []
for i in range(0, len(rsd_columns_mor)): 
    rsd = 100*np.std(records_prob_mor_cleaned[records_prob_mor_cleaned.columns[i]])/(np.mean(records_prob_mor_cleaned[records_prob_mor_cleaned.columns[i]])+0.00000000000000000000001)
    rsd_prob_mor.append(rsd)
data_rsd_mor.loc[0] = rsd_prob_mor


In [141]:
data_prob_mor = pd.concat([data_prob_mor, data_rsd_mor], axis=1)

# Afternoon data for smiling and eyes probabilities

In [142]:
COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
COLUMN_NAMES_aft = []
for i in range(0, len(COLUMN_NAMES)):
    COLUMN_NAMES_aft.append(COLUMN_NAMES[i] +'_aft')
records_prob_aft = pd.DataFrame(columns= COLUMN_NAMES_aft)

for i in range(0, len(data_afternoon)):
    prob_data = data_afternoon[i]['classification']
    prob_values = list(prob_data.values())

    if len(prob_data)!=0:
        records_prob_aft.loc[i] = prob_values
    else: 
        records_prob_aft.loc[i] = [np.nan]*3

In [143]:
records_prob_aft_cleaned = records_prob_aft.copy()
records_prob_aft_cleaned = records_prob_aft_cleaned.dropna()

In [144]:
if len(records_prob_aft_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
    cfg = tsfel.get_features_by_domain()
    # Extract features
    X = tsfel.time_series_features_extractor(cfg, records_prob_aft_cleaned)
    X_aft_to_delete = []
    for name in X.columns:
        if 'Spectrogram mean coefficient_' in name:
            X_aft_to_delete.append(name)
    X = X.drop(X_aft_to_delete, axis=1)
            
else:
    X = pd.DataFrame(columns=tself_columns_aft)
    X.loc[0] = [np.nan]*372

In [145]:
data_prob_aft = X.copy()

In [146]:
if len(records_prob_aft_cleaned)>3:        
    for j in range(0, len(COLUMN_NAMES_aft)):
        name_prob = COLUMN_NAMES_aft[j]
        features_pycatch = pycatch22.catch22_all(records_prob_aft_cleaned[name_prob])
        COLUMN = []
        for i in range(0, len(features_pycatch['names'])):
            COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
        features_prob_sub_aft = pd.DataFrame(columns=COLUMN)
        features_prob_sub_aft.loc[0] = features_pycatch['values']
        if j == 0:
            features_prob_aft = features_prob_sub_aft.copy()
        else:
            features_prob_aft = pd.concat([features_prob_aft, features_prob_sub_aft], axis=1)
else:
    features_prob_aft = pd.DataFrame(columns=pycatch_columns_aft)
    features_prob_aft.loc[0] = [np.nan]*66 



In [147]:
data_prob_aft = pd.concat([data_prob_aft, features_prob_aft], axis=1)

In [148]:
approx_entropy_columns = [name + '_app_ent' for name in records_prob_aft_cleaned.columns]
data_approx_entropy_aft = pd.DataFrame(columns=approx_entropy_columns)
app_ent_prob_aft= []



In [149]:
for i in range(0, len(approx_entropy_columns)): 
    try:
        approximate_entropy, parameters = nk.entropy_approximate(records_prob_aft_cleaned[records_prob_aft_cleaned.columns[i]])
         # Approximate entropy
    except:
        approximate_entropy = 0
    app_ent_prob_aft.append(approximate_entropy)
data_approx_entropy_aft.loc[0] = app_ent_prob_aft

In [150]:
data_approx_entropy_aft

,lefteye_aft_app_ent,righteye_aft_app_ent,smiling_aft_app_ent
0,0.389804,0.184115,0.117783


In [151]:
data_prob_aft = pd.concat([data_prob_aft, data_approx_entropy_aft], axis=1)

In [152]:
rsd_columns_aft= [name + '_aft' for name in records_prob_aft_cleaned.columns]
data_rsd_aft = pd.DataFrame(columns=rsd_columns_aft)
rsd_prob_aft = []
for i in range(0, len(rsd_columns_aft)): 
    rsd = 100*np.std(records_prob_aft_cleaned[records_prob_aft_cleaned.columns[i]])/(np.mean(records_prob_aft_cleaned[records_prob_aft_cleaned.columns[i]])+0.00000000000000000000001)
    rsd_prob_aft.append(rsd)
data_rsd_aft.loc[0] = rsd_prob_aft


In [153]:
data_prob_aft = pd.concat([data_prob_aft, data_rsd_aft], axis=1)

# Probabilities for evening

In [154]:
COLUMN_NAMES = ['lefteye', 'righteye', 'smiling']
COLUMN_NAMES_eve = []
for i in range(0, len(COLUMN_NAMES)):
    COLUMN_NAMES_eve.append(COLUMN_NAMES[i] +'_eve')
records_prob_eve = pd.DataFrame(columns= COLUMN_NAMES_eve)

for i in range(0, len(data_evening)):
    prob_data = data_evening[i]['classification']
    prob_values = list(prob_data.values())

    if len(prob_data)!=0:
        records_prob_eve.loc[i] = prob_values
    else: 
        records_prob_eve.loc[i] = [np.nan]*3

In [155]:
records_prob_eve_cleaned = records_prob_eve.copy()
records_prob_eve_cleaned = records_prob_eve_cleaned.dropna()

In [156]:
if len(records_prob_eve_cleaned) >=12:     # Retrieves a pre-defined feature configuration file to extract the temporal, statistical and spectral feature sets
    cfg = tsfel.get_features_by_domain()

    # Extract features
    X = tsfel.time_series_features_extractor(cfg, records_prob_eve_cleaned)
    X_eve_to_delete = []
    for name in X.columns:
        if 'Spectrogram mean coefficient_' in name:
            X_eve_to_delete.append(name)
    X = X.drop(X_eve_to_delete, axis=1)
            
else:
    X = pd.DataFrame(columns=tself_columns_eve)
    X.loc[0] = [np.nan]*372

In [157]:
data_prob_eve = X.copy()

In [158]:
if len(records_prob_eve_cleaned)>3:    
    for j in range(0, len(COLUMN_NAMES_eve)):
        name_prob = COLUMN_NAMES_eve[j]
        features_pycatch = pycatch22.catch22_all(records_prob_eve_cleaned[name_prob])
        COLUMN = []
        for i in range(0, len(features_pycatch['names'])):
            COLUMN.append(name_prob+ '_' + features_pycatch['names'][i]) 
        features_prob_sub_eve = pd.DataFrame(columns=COLUMN)
        features_prob_sub_eve.loc[0] = features_pycatch['values']
        if j == 0:
            features_prob_eve = features_prob_sub_eve.copy()
        else:
            features_prob_eve = pd.concat([features_prob_eve, features_prob_sub_eve], axis=1)
else:
    features_prob_eve = pd.DataFrame(columns=pycatch_columns_eve)
    features_prob_eve.loc[0] = [np.nan]*66 



In [159]:
data_prob_eve = pd.concat([data_prob_eve, features_prob_eve], axis=1)

In [160]:
approx_entropy_columns = [name + '_app_ent' for name in records_prob_eve_cleaned.columns]
data_approx_entropy_eve = pd.DataFrame(columns=approx_entropy_columns)
app_ent_prob_eve= []



In [161]:
for i in range(0, len(approx_entropy_columns)): 
    try:
        approximate_entropy, parameters = nk.entropy_approximate(records_prob_eve_cleaned[records_prob_eve_cleaned.columns[i]])
         # Approximate entropy
    except:
        approximate_entropy = 0
    app_ent_prob_eve.append(approximate_entropy)
data_approx_entropy_eve.loc[0] = app_ent_prob_eve
data_approx_entropy_eve

,lefteye_eve_app_ent,righteye_eve_app_ent,smiling_eve_app_ent
0,0,0,0


In [162]:
data_prob_eve = pd.concat([data_prob_eve, data_approx_entropy_eve], axis=1)

In [163]:
rsd_columns_eve= [name + '_rsd' for name in records_prob_eve_cleaned.columns]
data_rsd_eve = pd.DataFrame(columns=rsd_columns_eve)
rsd_prob_eve = []
for i in range(0, len(rsd_columns_eve)): 
    rsd = 100*np.std(records_prob_eve_cleaned[records_prob_eve_cleaned.columns[i]])/(np.mean(records_prob_eve_cleaned[records_prob_eve_cleaned.columns[i]])+0.00000000000000000000001)
    rsd_prob_eve.append(rsd)
data_rsd_eve.loc[0] = rsd_prob_eve


In [164]:
data_prob_eve = pd.concat([data_prob_eve, data_rsd_eve], axis=1)

# Concat all probabilites

In [165]:
data_prob = pd.concat([data_prob_mid, data_prob_mor, data_prob_aft, data_prob_eve], axis=1)

In [166]:
information_record = pd.DataFrame(columns = ['patient', 'record', 'diagnosis', 'timestamp_start', 'timestamp_end'])

In [167]:
information_record.loc[0] = [patient, record, diagnosis, timestamp_start, timestamp_end]

In [168]:
data_prob = pd.concat([information_record, data_prob], axis=1, join='inner')